# Lab 5, Absolutist Keyword Spotting, Training

Builds and trains a keyword spotting model on the merged dataset produced by
the data-construction notebook (`myKWSDataset.zip`), converts it to
TensorFlow Lite, and exports a C header ready for TensorFlow Lite for
Microcontrollers on the Nano 33 BLE Sense.

There are 7 classes, the 5 absolutist target words, plus `_unknown_` (other
spoken words, so the model learns to reject speech it was not trained on)
and `_silence_` (background noise, so the model learns to reject
non-speech).

In [ ]:
import os
import glob
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

SAMPLE_RATE = 16000
CLIP_LEN = SAMPLE_RATE  # 1 second clips, matches the dataset-construction notebook's output

## Upload the dataset

Upload `myKWSDataset.zip`, the file produced by the dataset-construction
notebook. It already contains `dataset/never`, `dataset/none`, `dataset/all`,
`dataset/must`, `dataset/only`, `dataset/_unknown_`, and
`dataset/_background_noise_`.

In [ ]:
from google.colab import files

uploaded = files.upload()
!unzip -oq myKWSDataset.zip
!ls dataset

## Build the `_silence_` class

The dataset does not have a silence class yet, only raw background noise
clips (each several seconds to a couple of minutes long). This cell cuts
those into 1 second crops, the same duration as every other class, so
`_silence_` can be trained on the same footing as the spoken-word classes.
The model needs this class to learn to report nothing when there is no
speech at all, not just when there is unrecognized speech (`_unknown_`).

In [ ]:
SILENCE_DIR = 'dataset/_silence_'
os.makedirs(SILENCE_DIR, exist_ok=True)

NUM_SILENCE_CLIPS = 260  # comparable order of magnitude to _unknown_ (315), so neither class dominates

noise_files = glob.glob('dataset/_background_noise_/*.wav')
print(f'Found {len(noise_files)} background noise source files.')

rng = np.random.RandomState(SEED)
clip_i = 0
for _ in range(NUM_SILENCE_CLIPS):
    src_path = noise_files[rng.randint(len(noise_files))]
    audio_binary = tf.io.read_file(src_path)
    audio, sr = tf.audio.decode_wav(audio_binary, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1).numpy()
    if len(audio) <= CLIP_LEN:
        crop = np.pad(audio, (0, CLIP_LEN - len(audio)))
    else:
        start = rng.randint(0, len(audio) - CLIP_LEN)
        crop = audio[start:start + CLIP_LEN]
    out_path = os.path.join(SILENCE_DIR, f'silence_{clip_i:04d}.wav')
    tf.io.write_file(out_path, tf.audio.encode_wav(crop.reshape(-1, 1).astype(np.float32), SAMPLE_RATE))
    clip_i += 1

print(f'Wrote {clip_i} silence clips to {SILENCE_DIR}')

## Explore the dataset

Final per-class counts, worth recording directly in the report's Section B
(dataset size and instances per class).

In [ ]:
CLASSES = ['_silence_', '_unknown_', 'all', 'must', 'never', 'none', 'only']

class_files = {}
for c in CLASSES:
    files_c = sorted(glob.glob(f'dataset/{c}/*.wav'))
    class_files[c] = files_c
    print(f'{c}: {len(files_c)} clips')

total = sum(len(v) for v in class_files.values())
print(f'\nTotal: {total} clips across {len(CLASSES)} classes')

## Train / validation / test split

Split by file, not by class-wide shuffling, using a fixed seed so the split
is reproducible. Each recording here is already an independent utterance or
independent crop (not repeated readings of one continuous trial the way the
IMU labs' rows were), so a stratified random split per class is appropriate,
unlike the trial-level split used in Labs 3 and 4.

In [ ]:
filepaths, labels = [], []
for label_idx, c in enumerate(CLASSES):
    for f in class_files[c]:
        filepaths.append(f)
        labels.append(label_idx)

filepaths = np.array(filepaths)
labels = np.array(labels)

train_files, temp_files, train_labels, temp_labels = train_test_split(
    filepaths, labels, test_size=0.30, random_state=SEED, stratify=labels)
val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files, temp_labels, test_size=0.50, random_state=SEED, stratify=temp_labels)

print(f'Train: {len(train_files)}  Val: {len(val_files)}  Test: {len(test_files)}')

## Feature extraction, log-magnitude spectrogram

Each 1 second, 16 kHz clip becomes a 2D time-frequency "image" via a short
time Fourier transform, the standard feature representation for small audio
CNNs.

This is deliberately a raw STFT magnitude spectrogram rather than a mel
filterbank on top of it, kept intentionally simple since the exact same
feature computation has to be reproduced on the Arduino later. A full
mel-filterbank pipeline would add another matrix multiply and set of
constants to replicate in firmware for comparatively little accuracy
benefit at this model size.

`frame_length`/`frame_step`/the frequency-bin cropping below were sized
specifically for the Nano 33 BLE Sense's memory, not just picked for a
clean output shape. TensorFlow Lite for Microcontrollers has to hold a
layer's input and output activations in RAM at the same time during
inference, and an earlier attempt at 124 time frames x 129 frequency bins
made the first convolution's activation alone work out to roughly 496 KB,
almost double the board's entire 256 KB of RAM, a completely separate
budget from the model file's flash size, and a hard failure no matter how
the tensor arena is sized. `frame_step=400` cuts the number of time frames
to 40, and cropping to the bottom 65 frequency bins (0-4 kHz, out of a full
range to 8 kHz) keeps only where target speech energy actually concentrates
rather than needing every bin. `frame_length` stays at 256, wide enough for
reasonable frequency resolution, the crop shrinks the bin count without
sacrificing that resolution.

In [ ]:
FRAME_LENGTH = 256
FRAME_STEP = 400
MAX_FREQ_BIN = 65  # crop to 0-4 kHz (fft_length=256 gives 62.5 Hz per bin)

def load_and_pad(path):
    audio_binary = tf.io.read_file(path)
    audio, sr = tf.audio.decode_wav(audio_binary, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)
    audio_len = tf.shape(audio)[0]
    audio = tf.cond(
        audio_len < CLIP_LEN,
        lambda: tf.pad(audio, [[0, CLIP_LEN - audio_len]]),
        lambda: audio[:CLIP_LEN],
    )
    # Peak-normalize so clip-to-clip loudness differences don't confound
    # the spectrogram. This matters a lot here specifically, the 5 target
    # words were recorded through a browser mic with no leveling, while
    # _unknown_/_background_noise_ come from the professionally leveled
    # Speech Commands set, so without this the model partly learns to key
    # off loudness (which differs mostly by SOURCE, not by word) rather
    # than the actual spoken pattern. Guarded against near-silent clips so
    # this doesn't amplify background hiss into loud noise.
    peak = tf.reduce_max(tf.abs(audio))
    audio = tf.cond(peak > 1e-3, lambda: audio / peak, lambda: audio)
    return audio

def get_spectrogram(audio):
    stft = tf.signal.stft(audio, frame_length=FRAME_LENGTH, frame_step=FRAME_STEP)
    stft = stft[:, :MAX_FREQ_BIN]
    spectrogram = tf.abs(stft)
    spectrogram = tf.math.log(spectrogram + 1e-6)
    return spectrogram

def path_to_features(path, label):
    audio = load_and_pad(path)
    spectrogram = get_spectrogram(audio)
    spectrogram = tf.expand_dims(spectrogram, axis=-1)  # add channel dim
    return spectrogram, label

# Sanity check on one example, confirms the shape we expect before building
# the full pipeline.
example_spec, example_label = path_to_features(train_files[0], train_labels[0])
print('Spectrogram shape:', example_spec.shape, ' Class:', CLASSES[example_label])

plt.figure(figsize=(6, 4))
plt.imshow(tf.squeeze(example_spec).numpy().T, aspect='auto', origin='lower')
plt.title(f'Example spectrogram, class={CLASSES[example_label]}')
plt.xlabel('Time frame')
plt.ylabel('Frequency bin')
plt.colorbar()
plt.tight_layout()
plt.savefig('example_spectrogram.png', dpi=150)
plt.show()

## Normalization

Same reasoning as Labs 3 and 4, compute a fixed (mean, std) from the
training set only, apply it identically to validation, test, and later
real-time inference on the board. Since every spectrogram bin here has
comparable scale and meaning (log-magnitude at a given frequency), a single
global scalar mean/std is used rather than 128 separate per-bin constants,
keeping the on-device normalization to two numbers instead of 256.

In [ ]:
def compute_norm_stats(files, sample_size=200):
    sample_files = files if len(files) <= sample_size else np.random.RandomState(SEED).choice(files, sample_size, replace=False)
    values = []
    for f in sample_files:
        spec, _ = path_to_features(f, 0)
        values.append(spec.numpy().flatten())
    values = np.concatenate(values)
    return float(values.mean()), float(values.std())

SPEC_MEAN, SPEC_STD = compute_norm_stats(train_files)
print(f'SPEC_MEAN = {SPEC_MEAN:.6f}')
print(f'SPEC_STD  = {SPEC_STD:.6f}')

def normalize(spectrogram, label):
    return (spectrogram - SPEC_MEAN) / SPEC_STD, label

## Build the `tf.data` pipelines

In [ ]:
def make_dataset(files, labels, shuffle=False, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices((files, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(files), seed=SEED)
    ds = ds.map(path_to_features, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_files, train_labels, shuffle=True)
val_ds = make_dataset(val_files, val_labels)
test_ds = make_dataset(test_files, test_labels)

INPUT_SHAPE = example_spec.shape
NUM_CLASSES = len(CLASSES)
print('Input shape:', INPUT_SHAPE, ' Num classes:', NUM_CLASSES)

## Model architecture

A small CNN over the spectrogram, two convolution/pooling blocks followed by
a dense classifier. Dropout and early stopping are carried forward from the
overfitting fix introduced in Lab 4 (itself a response to the overfitting
Lab 3 documented), rather than rediscovering that lesson from scratch.

The pooled feature maps are reduced with `GlobalAveragePooling2D` rather
than `Flatten`, deliberately, not just as a style choice. The spectrogram
input (124x129) is far larger than Lab 4's 45-value IMU windows, so
flattening it before the first dense layer would create roughly 445,000
parameters in that one layer alone, a model over 1.7 MB, bigger than the
Nano 33 BLE Sense's entire 1 MB flash budget before even the sketch code and
TFLite Micro library are added.

A first attempt paired `GlobalAveragePooling2D` with only two conv blocks
(8 then 16 filters) and it was too aggressive, averaging away almost all of
the timing/frequency pattern that actually distinguishes similarly-shaped
words, collapsing every word class's accuracy (one class dropped to 0
percent correct) while `_silence_`/`_unknown_` stayed fine, since those two
are easy, broad categories, not fine-grained word discrimination. Training
accuracy itself plateaued in the 70s, confirming underfitting, not just a
train/val generalization gap. The fix keeps the same pooling strategy but
gives the network more depth and channels beforehand, a third conv block
(32 filters) before the average pooling, so more of that pattern survives
into the final 32-value feature vector instead of being averaged away too
early.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=INPUT_SHAPE),
    tf.keras.layers.Conv2D(8, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## Train

`_silence_` (182 train clips) and `_unknown_` (220) heavily outnumber each
individual word (about 35 each), so without correction the loss function
can rack up high accuracy just by nailing the two big classes and barely
try on the words. `class_weight` counteracts this directly, weighting each
class inversely to how often it appears in training, so a mistake on a
rare word class counts for more than a mistake on `_silence_`/`_unknown_`.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weight_values = compute_class_weight(
    class_weight='balanced', classes=np.arange(NUM_CLASSES), y=train_labels)
class_weight = dict(enumerate(class_weight_values))
for i, c in enumerate(CLASSES):
    print(f'{c}: weight={class_weight[i]:.2f}')

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,
    callbacks=[early_stop],
    class_weight=class_weight,
)

## Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('curves.png', dpi=150)
plt.show()

## Evaluate on the test set

Overall accuracy, plus a per-class breakdown and confusion matrix, since a
single aggregate number would hide whether errors are concentrated in one
or two confusable classes (as they were with sitting/unknown in Lab 4).

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f'Overall test accuracy: {test_acc:.4f}')

y_true, y_pred = [], []
for spec_batch, label_batch in test_ds:
    preds = model.predict(spec_batch, verbose=0)
    y_true.extend(label_batch.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print('\nPer-class accuracy:')
for i, c in enumerate(CLASSES):
    mask = y_true == i
    if mask.sum() > 0:
        acc = (y_pred[mask] == i).mean()
        print(f'  {c}: {acc:.3f}  ({mask.sum()} test clips)')

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(NUM_CLASSES), CLASSES, rotation=45, ha='right')
plt.yticks(range(NUM_CLASSES), CLASSES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Test confusion matrix')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                  color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar()
plt.tight_layout()
plt.savefig('confusion.png', dpi=150)
plt.show()

## Convert to TensorFlow Lite

Float32 conversion, same choice made in Lab 4, this model is already small
enough to fit comfortably in the board's flash and RAM without needing
int8 quantization's added complexity (a representative dataset and a
calibration pass).

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f'Saved model.tflite, {len(tflite_model)} bytes')

## Export as a C header

Same pattern as Lab 4, a byte array (`g_model`/`g_model_len`) ready to
embed directly in the Arduino sketch via TensorFlow Lite for
Microcontrollers, plus the normalization constants and feature parameters
the firmware needs to reproduce the same spectrogram computation used here.

In [ ]:
def write_c_header(tflite_path, header_path, array_name='g_model'):
    with open(tflite_path, 'rb') as f:
        data = f.read()

    with open(header_path, 'w') as f:
        f.write(f'// Auto-generated from {tflite_path}, do not edit by hand.\n')
        f.write('#include <cstdint>\n\n')
        f.write(f'alignas(8) const unsigned char {array_name}[] = {{\n')
        for i in range(0, len(data), 12):
            chunk = data[i:i + 12]
            line = ', '.join(f'0x{b:02x}' for b in chunk)
            f.write(f'  {line},\n')
        f.write('};\n')
        f.write(f'const unsigned int {array_name}_len = {len(data)};\n')

    print(f'Wrote {header_path}, {len(data)} bytes as {array_name}')

write_c_header('model.tflite', 'model_data.h')

print()
print(f'const float SPEC_MEAN = {SPEC_MEAN:.6f}f;')
print(f'const float SPEC_STD = {SPEC_STD:.6f}f;')
print(f'const int SAMPLE_RATE = {SAMPLE_RATE};')
print(f'const int CLIP_LEN = {CLIP_LEN};')
print(f'const int FRAME_LENGTH = {FRAME_LENGTH};')
print(f'const int FRAME_STEP = {FRAME_STEP};')
print(f'// CLASSES order: {CLASSES}')

## Download

Download `model_data.h`, `model.tflite`, `curves.png`, `confusion.png`, and
`example_spectrogram.png` for use in the Arduino sketch and the report.

In [ ]:
from google.colab import files as colab_files

for fname in ['model_data.h', 'model.tflite', 'curves.png', 'confusion.png', 'example_spectrogram.png']:
    colab_files.download(fname)